# Black Sigatoka Feature Engineering Review

This review checks the existing synthetic train and validation files and documents the features already present. It does not change data or train models.

## 1. Load Dataset



In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from black_sigatoka_preprocessing import (
    FEATURES, TARGET, load_and_validate_datasets, prepare_model_inputs,
)

# Load revised training and validation data only
frames, summaries = load_and_validate_datasets(
    PROJECT_ROOT, splits=("train", "validation")
)
assert set(frames) == {"train", "validation"}
print("Train records:", len(frames["train"]))
print("Validation records:", len(frames["validation"]))
display(pd.DataFrame(summaries).T[["rows", "input_shape", "selected_missing_values", "selected_nonfinite_values"]])


Train records: 4501
Validation records: 937


,rows,input_shape,selected_missing_values,selected_nonfinite_values
train,4501,4501 x 3,0,0
validation,937,937 x 3,0,0


## 2. Available Features



In [2]:
# Check columns and completeness in each modeling partition
all_columns = frames["train"].columns.tolist()
available = pd.DataFrame({
    "Column": all_columns,
    "Train missing": [frames["train"][c].isna().sum() for c in all_columns],
    "Validation missing": [frames["validation"][c].isna().sum() for c in all_columns],
    "Train dtype": [frames["train"][c].dtype for c in all_columns],
    "Model input": [c in FEATURES for c in all_columns],
    "Target": [c == TARGET for c in all_columns],
})
display(available)


,Column,Train missing,Validation missing,Train dtype,Model input,Target
0,scenario_id,0,0,str,False,False
1,record_type,0,0,str,False,False
2,timestamp_utc,4501,937,float64,False,False
3,air_temp_c,0,0,float64,True,False
4,relative_humidity_pct,0,0,float64,False,False
5,rainfall_mm_hr,0,0,float64,False,False
6,dew_point_c,0,0,float64,False,False
7,dew_point_depression_c,0,0,float64,False,False
8,wind_speed_m_s,0,0,float64,False,False
9,solar_radiation_mj_hr,0,0,float64,False,False


## 3. Current Model Features



In [3]:
# Check the approved feature values
model_inputs, targets = prepare_model_inputs(frames, splits=("train", "validation"))
X_train, X_val = model_inputs["train"], model_inputs["validation"]
y_train, y_val = targets["train"], targets["validation"]
feature_summary = pd.DataFrame({
    "Feature": FEATURES,
    "Train missing": [X_train[c].isna().sum() for c in FEATURES],
    "Validation missing": [X_val[c].isna().sum() for c in FEATURES],
    "Train minimum": [X_train[c].min() for c in FEATURES],
    "Train maximum": [X_train[c].max() for c in FEATURES],
    "Validation minimum": [X_val[c].min() for c in FEATURES],
    "Validation maximum": [X_val[c].max() for c in FEATURES],
})
print("Target:", TARGET)
display(feature_summary)


Target: risk_score


,Feature,Train missing,Validation missing,Train minimum,Train maximum,Validation minimum,Validation maximum
0,air_temp_c,0,0,17.011,33.866,17.029,33.988
1,estimated_leaf_wet,0,0,0.000,1.000,0.000,1.000
2,consecutive_wetness_hours,0,0,0.000,120.000,0.000,120.000


## 4. Check Engineered Features

The selected wetness-duration feature is already present. The project builder copied `wetness_duration_hours` from the generated scenario. In separate simulated full-feature files, 24/72-hour wet-hour summaries were set to the smaller of that duration and the window size. These are synthetic scenario summaries, not rolling calculations from actual scenario timestamps.

In [4]:
# Review wetness duration and the existing window summaries
wetness_checks = []
for split, frame in frames.items():
    wet_flag_matches_duration = frame["estimated_leaf_wet"].eq(
        frame["consecutive_wetness_hours"].gt(0).astype(int)
    ).all()
    wetness_checks.append({
        "Split": split,
        "Wetness-duration missing": frame["consecutive_wetness_hours"].isna().sum(),
        "Wetness-duration minimum": frame["consecutive_wetness_hours"].min(),
        "Wetness-duration maximum": frame["consecutive_wetness_hours"].max(),
        "Wet flag equals duration > 0": wet_flag_matches_duration,
        "24h summary equals min(duration, 24)": frame["wet_hours_previous_24h"].eq(
            frame["consecutive_wetness_hours"].clip(upper=24)
        ).all(),
        "72h summary equals min(duration, 72)": frame["wet_hours_previous_72h"].eq(
            frame["consecutive_wetness_hours"].clip(upper=72)
        ).all(),
    })
display(pd.DataFrame(wetness_checks))

# Observation timestamps are blank in the synthetic scenarios
print("Train scenario timestamps present:", pd.to_datetime(
    frames["train"]["timestamp_utc"], utc=True, errors="coerce"
).notna().sum())
print("Validation scenario timestamps present:", pd.to_datetime(
    frames["validation"]["timestamp_utc"], utc=True, errors="coerce"
).notna().sum())
print("Wetness method:", frames["train"]["wetness_estimation_method"].unique().tolist())


,Split,Wetness-duration missing,Wetness-duration minimum,Wetness-duration maximum,Wet flag equals duration > 0,"24h summary equals min(duration, 24)","72h summary equals min(duration, 72)"
0,train,0,0,120,True,True,True
1,validation,0,0,120,True,True,True


Train scenario timestamps present: 0
Validation scenario timestamps present: 0
Wetness method: ['SCENARIO_SET_RH_GE_98_WET_ELSE_DRY; NOT SENSOR']


## 5. Check Missing Values



In [5]:
# Check completeness of current and proposed features
review_names = [
    "air_temp_c", "estimated_leaf_wet", "consecutive_wetness_hours",
    "wet_hours_previous_24h", "wet_hours_previous_72h",
    "mean_temp_previous_24h_c", "rain_previous_24h_mm",
    "relative_humidity_pct", "canopy_temperature_c", "dew_point_depression_c",
]
missing_rows = []
for name in review_names:
    missing_rows.append({
        "Feature": name,
        "Exists": name in frames["train"].columns,
        "Train missing": int(frames["train"][name].isna().sum()) if name in frames["train"] else None,
        "Validation missing": int(frames["validation"][name].isna().sum()) if name in frames["validation"] else None,
    })
missing_table = pd.DataFrame(missing_rows)
display(missing_table)


,Feature,Exists,Train missing,Validation missing
0,air_temp_c,True,0,0
1,estimated_leaf_wet,True,0,0
2,consecutive_wetness_hours,True,0,0
3,wet_hours_previous_24h,True,0,0
4,wet_hours_previous_72h,True,0,0
5,mean_temp_previous_24h_c,True,4501,937
6,rain_previous_24h_mm,True,4501,937
7,relative_humidity_pct,True,0,0
8,canopy_temperature_c,True,4501,937
9,dew_point_depression_c,True,0,0


## 6. Feature Relationships

The correlations below are descriptive associations with synthetic scores. Correlation alone cannot show that a feature is useful, causal, or free of leakage.

In [6]:
# Compare the three selected features with the synthetic target
correlations = pd.concat([X_train, y_train.rename(TARGET)], axis=1).corr(numeric_only=True)[TARGET]
display(correlations.rename("Correlation with synthetic risk score").to_frame())

# Find repeated inputs with differing scores in train and validation
review_frame = pd.concat([frames["train"], frames["validation"]], ignore_index=True)
input_score_groups = review_frame.groupby(FEATURES, dropna=False)[TARGET].agg(
    rows="size", distinct_scores="nunique", score_min="min", score_max="max"
)
conflicting_groups = input_score_groups[input_score_groups["distinct_scores"] > 1]
print("Repeated three-feature groups with different scores:", len(conflicting_groups))
display(conflicting_groups.head(10))


,Correlation with synthetic risk score
air_temp_c,0.363210
estimated_leaf_wet,0.528492
consecutive_wetness_hours,0.719605
risk_score,1.000000


Repeated three-feature groups with different scores: 5


,,,rows,distinct_scores,score_min,score_max
air_temp_c,estimated_leaf_wet,consecutive_wetness_hours,,,,
20.624,1,108,2,2,60.8005,60.8106
22.185,1,20,2,2,11.6491,11.6515
22.604,1,11,2,2,5.1169,5.1175
23.297,1,37,2,2,44.4125,44.4167
28.929,1,47,2,2,75.6192,75.6225


## 7. Feature Engineering Decisions

A sensor wetness streak should be calculated after sorting readings by sensor/leaf and timestamp: increase the count for each consecutive wet hourly reading, reset to zero on a dry reading, and handle missing hours with a documented gap rule. The streak must use only readings available at the prediction time. A leaf-wetness sensor measures a surface wetness response; `estimated_leaf_wet` in these files is a synthetic indicator, not that measurement.

In [7]:
decision_rows = [
    ("air_temp_c", "Synthetic environmental input / proxy", "Yes", "Complete and physically measurable by an IoT air-temperature sensor; current project rows are synthetic, not field readings."),
    ("estimated_leaf_wet", "Estimated binary indicator", "Yes", "Complete, but assigned by the synthetic scenario rule; not a leaf-wetness sensor measurement."),
    ("consecutive_wetness_hours", "Engineered duration feature", "Yes", "Complete and relevant to the synthetic score rule; current scenario rows do not contain a chronological wetness series to independently rebuild it."),
    ("wet_hours_previous_24h", "Engineered / scenario summary", "No", "Complete but set from the synthetic duration cap; redundant and not a real trailing-window calculation in these scenarios."),
    ("wet_hours_previous_72h", "Engineered / scenario summary", "No", "Complete but set from the synthetic duration cap; redundant and not a real trailing-window calculation in these scenarios."),
    ("mean_temp_previous_24h_c", "Engineered rolling summary", "No", "All synthetic modeling values are missing; it is not a mean temperature during wet periods."),
    ("rain_previous_24h_mm", "Engineered rolling summary", "No", "All synthetic modeling values are missing."),
    ("relative_humidity_pct", "Simulated in scenarios; measured by RH sensor in IoT", "No", "Complete but constructed from wet/dry scenario state, strongly tied to estimated wetness; possible synthetic shortcut."),
    ("canopy_temperature_c", "Measured/estimated candidate", "No", "All synthetic and available historical-file values are missing; a canopy sensor would be needed."),
    ("dew_point_depression_c", "Derived environmental feature", "No", "Complete but derived from air/dew-point temperatures; redundant with those inputs and the synthetic humidity setup."),
]
decisions = pd.DataFrame(decision_rows, columns=["Feature", "Raw / Estimated / Engineered", "Used in Model", "Reason"])
display(decisions)

print("Existing EDA rolling audit: historical 24/72-hour summaries matched windows including the current hour; previous-only windows did not match.")
print("No data rows were changed and no model was trained.")


,Feature,Raw / Estimated / Engineered,Used in Model,Reason
0,air_temp_c,Synthetic environmental input / proxy,Yes,Complete and physically measurable by an IoT a...
1,estimated_leaf_wet,Estimated binary indicator,Yes,"Complete, but assigned by the synthetic scenar..."
2,consecutive_wetness_hours,Engineered duration feature,Yes,Complete and relevant to the synthetic score r...
3,wet_hours_previous_24h,Engineered / scenario summary,No,Complete but set from the synthetic duration c...
4,wet_hours_previous_72h,Engineered / scenario summary,No,Complete but set from the synthetic duration c...
5,mean_temp_previous_24h_c,Engineered rolling summary,No,All synthetic modeling values are missing; it ...
6,rain_previous_24h_mm,Engineered rolling summary,No,All synthetic modeling values are missing.
7,relative_humidity_pct,Simulated in scenarios; measured by RH sensor ...,No,Complete but constructed from wet/dry scenario...
8,canopy_temperature_c,Measured/estimated candidate,No,All synthetic and available historical-file va...
9,dew_point_depression_c,Derived environmental feature,No,Complete but derived from air/dew-point temper...


Existing EDA rolling audit: historical 24/72-hour summaries matched windows including the current hour; previous-only windows did not match.
No data rows were changed and no model was trained.
